# Approach 2 · A5 + Paired Disease Patch Supervision
## 1. Setup

Run with the team requirements on Python 3.13.17 and CUDA 12.8. Use `DEBUG=True` for a train-only smoke test. Full runs save a unique experiment folder and an environment snapshot.

The historical A5 achieved 23.48% proxy tune Top-1 and 23.85% independent proxy check Top-1, then 8.49% on the official joint test. The experiment below tests a change to the disease representation during training with raw scoring as the primary endpoint.

In [ ]:
DEBUG = False

from datetime import datetime, timezone
from pathlib import Path
from importlib import metadata
import platform
import copy
import hashlib
import json
import sys

if sys.version_info[:3] != (3, 13, 17):
    raise RuntimeError(f"Select the Python 3.13.17 notebook kernel. Current kernel: {sys.executable}.")

import pandas as pd
from packaging.requirements import Requirement
from packaging.utils import canonicalize_name
import timm
import torch
from IPython.display import display
from PIL import Image
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision.transforms import functional as TF

here = Path.cwd()
ROOT = next((p for p in (here, *here.parents) if (p / "common" / "dataset.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Open the notebook inside the team repository.")
sys.path.insert(0, str(ROOT))

from common import config as team
from common.dataset import PlantDataset, get_loaders, get_pairs, get_transforms, load_split
from common.evaluate import PERTURBATIONS, harmonic_mean, save_confusion, top_k_accuracy

BACKBONE = "vit_base_patch14_dinov2.lvd142m"
REFERENCE_COMMIT = "0fece2daff4917431bb2a5b16b65718ab100d05c"
DEVICE = team.DEVICE
NUM_CROPS, NUM_DISEASES = team.NUM_CROPS, team.NUM_DISEASES
team.NUM_WORKERS = 4
BATCH_SIZE, EPOCHS = (8, 1) if DEBUG else (16, 20)
HEAD_LR, BACKBONE_LR, WEIGHT_DECAY = 1e-4, 1e-5, 1e-4
SMOOTHING = 0.1
DECOR_WEIGHT, SUPCON_WEIGHT, TEMPERATURE = 0.05, 0.10, 0.10
BLUR_PROB, BLUR_SIGMA = 0.15, 0.7
LOGIT_SCALE = 16.0
HARD_NEGATIVES, PAIR_MARGIN, RANK_WEIGHT, TRANSFER_WEIGHT = 3, 1.0, 0.35, 0.25
RUN_NAME = "approach2_a5_paired_attention" + ("_debug" if DEBUG else "")
CONTROL, PAIRED = "A5_Control", "A5_PairedDiseaseAttention"
RECIPES = {CONTROL: False, PAIRED: True}
PATCH_GRID, DONOR_TEMPERATURE = 8, 0.1
PAIRED_WEIGHT, ALIGNMENT_WEIGHT = 0.3, 0.1
GAMMAS = [step / 2 for step in range(25)]
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
RESULTS_DIR = team.RESULTS_DIR / RUN_NAME / RUN_ID
DEVELOPMENT_DIR = RESULTS_DIR / "development"
FINAL_DIR = RESULTS_DIR / "final"
OFFICIAL_DIR = RESULTS_DIR / "official"
CHECKPOINT = team.CHECKPOINT_DIR / RUN_NAME / RUN_ID / "model.pt"
LOCK = FINAL_DIR / "selection_lock.json"

requirements_path = ROOT / "requirements.txt"
requirements_bytes = requirements_path.read_bytes()
requirements = requirements_bytes.decode("utf-8-sig")
for entry in requirements.splitlines():
    entry = entry.split(" #", 1)[0].strip()
    if not entry or entry.startswith(("#", "--index-url", "--extra-index-url")):
        continue
    requirement = Requirement(entry)
    if requirement.marker and not requirement.marker.evaluate():
        continue
    try:
        installed_version = metadata.version(requirement.name)
    except metadata.PackageNotFoundError as error:
        raise RuntimeError(f"Install the repository requirements in this kernel: {entry}.") from error
    if not requirement.specifier.contains(installed_version, prereleases=True):
        raise RuntimeError(f"requirements.txt specifies {entry}, but this kernel has {installed_version}.")

if not DEBUG and (DEVICE.type != "cuda" or torch.version.cuda != "12.8"
                  or not torch.cuda.is_available()):
    raise RuntimeError("The full run requires the project CUDA 12.8 PyTorch build and a visible GPU.")

environment = {
    "python": platform.python_version(),
    "platform": platform.platform(),
    "machine": platform.machine(),
    "executable": sys.executable,
    "requirements_sha256": hashlib.sha256(requirements_bytes).hexdigest(),
    "cuda_build": torch.version.cuda,
    "device": str(DEVICE),
    "gpu": torch.cuda.get_device_name(DEVICE) if DEVICE.type == "cuda" else None,
    "packages": dict(sorted((canonicalize_name(dist.metadata["Name"]), dist.version)
                            for dist in metadata.distributions() if dist.metadata["Name"])),
}

if not team.TRAIN_CSV.is_file() or not team.IMAGE_DIR.is_dir():
    raise FileNotFoundError("Training CSV or image directory is missing.")
if (team.NUM_SEEN_PAIRS, team.NUM_UNSEEN_PAIRS, team.NUM_PAIRS) != (323, 10, 333):
    raise ValueError("Use the shared 323/333 evaluation API.")
print(f"PyTorch {torch.__version__} · {DEVICE} · debug={DEBUG} · run {RUN_ID}")

## 2. Data

Use the original A5 split. Development holds out ten complete combinations from the training CSV while preserving each crop and disease concept in fit. Five held pairs guide selection and five support a separate check. The two arms use identical images, batches and augmentations. Every cross-crop donor is selected from the current fit batch. During full refit, all training rows are restored.

In [ ]:
def image_key(value):
    result = Path(str(value).strip().replace("\\", "/"))
    if not result.name or result.is_absolute() or ".." in result.parts or ":" in str(result):
        raise ValueError(f"Expected a relative image filename, got {value!r}.")
    return result.as_posix()


def sha256_file(path):
    with path.open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()


def atomic_json(path, value):
    temp = path.with_suffix(".json.tmp")
    temp.write_text(json.dumps(value, indent=2), encoding="utf-8")
    temp.replace(path)


def atomic_save(path, value):
    temp = path.with_suffix(".pt.tmp")
    torch.save(value, temp)
    temp.replace(path)


def proxy_split(fit_frame, val_frame):
    """Hold out complete training pairs from the development heads only."""
    grouped = fit_frame.groupby("pair_row", sort=True)
    rows = grouped.agg(n=("image", "size"), crop=("crop", "first"), disease=("disease", "first"))
    crops = rows.crop.value_counts().to_dict()
    diseases = rows.disease.value_counts().to_dict()
    eligible = rows[rows.n.ge(5)].sample(frac=1, random_state=team.SEED + 1)
    held = []
    for pair_row, row in eligible.iterrows():
        if crops[row.crop] > 1 and diseases[row.disease] > 1:
            held.append(int(pair_row))
            crops[row.crop] -= 1
            diseases[row.disease] -= 1
        if len(held) == 10:
            break
    if len(held) < 10:
        raise ValueError("Ten train-derived holdouts with fit support were not available.")
    seen = val_frame.loc[~val_frame.pair_row.isin(held)].sample(frac=1, random_state=team.SEED + 2)
    cut = len(seen) // 2
    frames = {
        "tune_seen": seen.iloc[:cut],
        "tune_proxy": train_full.loc[train_full.pair_row.isin(held[:5])],
        "check_seen": seen.iloc[cut:],
        "check_proxy": train_full.loc[train_full.pair_row.isin(held[5:])],
    }
    if any(group.empty for group in frames.values()):
        raise ValueError("Proxy validation needs seen and held-out images.")
    fit_mask = torch.ones(len(train_pairs), dtype=torch.bool)
    fit_mask[held] = False
    return fit_mask, frames


train_full = load_split(team.TRAIN_CSV)
train_full["image"] = train_full.image.map(image_key)
if not train_full.image.is_unique:
    raise ValueError("Duplicate training images.")
train_pairs = get_pairs()
if train_pairs.shape != (team.NUM_SEEN_PAIRS, 2):
    raise ValueError("Unexpected training pair table.")
train_pairs_device = train_pairs.to(DEVICE)
_, image_transform = get_transforms("imagenet")

settings = {
    "reference_commit": REFERENCE_COMMIT, "backbone": BACKBONE, "seed": team.SEED,
    "image_size": team.IMAGE_SIZE, "resize_size": team.RESIZE_SIZE, "val_fraction": team.VAL_FRACTION,
    "batch_size": BATCH_SIZE, "epochs": EPOCHS, "head_lr": HEAD_LR, "backbone_lr": BACKBONE_LR,
    "weight_decay": WEIGHT_DECAY, "optimizer": "AdamW", "schedule": "CosineAnnealingLR",
    "label_smoothing": SMOOTHING, "trainable_backbone": ["blocks.11", "norm"],
    "decor_weight": DECOR_WEIGHT, "supcon_weight": SUPCON_WEIGHT, "temperature": TEMPERATURE,
    "blur_probability": BLUR_PROB, "blur_sigma": BLUR_SIGMA,
    "scoring": "raw_additive_cosine", "logit_scale": LOGIT_SCALE,
    "ranking_weight": RANK_WEIGHT, "ranking_margin": PAIR_MARGIN, "hard_negatives": HARD_NEGATIVES,
    "transfer_ranking_weight": TRANSFER_WEIGHT,
    "pair_weighting": "inverse_sqrt_fit_pair_frequency",
    "train_csv_sha256": sha256_file(team.TRAIN_CSV),
    "recipes": RECIPES, "paired_weight": PAIRED_WEIGHT, "alignment_weight": ALIGNMENT_WEIGHT,
    "donor_temperature": DONOR_TEMPERATURE, "patch_grid": PATCH_GRID, "gamma_grid": GAMMAS,
    "requirements_sha256": environment["requirements_sha256"],
    "environment_sha256": hashlib.sha256(json.dumps(environment, sort_keys=True).encode()).hexdigest(),
    "python": sys.version.split()[0],
    "torch": str(torch.__version__), "timm": timm.__version__, "debug": DEBUG,
}

## 3. Model

**A5 Control** retains the original crop and disease heads, disease patch attention, training of the final DINOv2 block and norm, weighted pair-ranking loss, transfer ranking, Cross-crop SupCon and logit decorrelation.

**A5 Paired Disease Attention** adds one training objective. For same-disease images from different crops, pool spatial tokens to an 8×8 grid, then use cosine patch matches to emphasize regions shared across crops. The A5 disease head classifies the shared description. An alignment term trains the ordinary single-image attention output to follow it. Every candidate is scored by the sum of the learned crop and disease logits at inference. Single-image inference needs no donor.

The patch matching objective is adapted from [Learning Attention as Disentangler, CVPR 2023](https://arxiv.org/abs/2303.15111).

In [ ]:
@torch.no_grad()
def frozen_prefix(backbone, images):
    tokens = backbone.patch_embed(images)
    tokens = backbone._pos_embed(tokens)
    tokens = backbone.norm_pre(backbone.patch_drop(tokens))
    for block in backbone.blocks[:-1]:
        tokens = block(tokens)
    return tokens


def augment_blur(images, generator):
    selected = torch.rand(len(images), generator=generator) < BLUR_PROB
    if not selected.any():
        return images
    selected = selected.to(images.device)
    augmented = images.clone()
    augmented[selected] = TF.gaussian_blur(images[selected], kernel_size=[5, 5],
                                           sigma=[BLUR_SIGMA, BLUR_SIGMA])
    return augmented


def center_logits(logits):
    # Remove the per-image offset to which categorical predictions are invariant.
    return logits - logits.mean(dim=1, keepdim=True)


def decorrelation(crop, disease):
    def standardize(x):
        x = x - x.mean(0)
        return x / (x.square().mean(0) + 1e-4).sqrt()

    cross_correlation = standardize(crop).T @ standardize(disease) / len(crop)
    return cross_correlation.square().mean()


def cross_crop_supcon(embedding, crop, disease):
    z = F.normalize(embedding, dim=1)
    same_disease = disease[:, None].eq(disease[None, :])
    positives = same_disease & crop[:, None].ne(crop[None, :])
    negatives = ~same_disease
    similarity = z @ z.T / TEMPERATURE
    normalizer = similarity.masked_fill(~(positives | negatives), -1e9).logsumexp(dim=1)
    counts = positives.sum(dim=1)
    positive_mean = similarity.masked_fill(~positives, 0.0).sum(dim=1) / counts.clamp_min(1)
    usable = (counts > 0) & negatives.any(dim=1)
    per_anchor = (normalizer - positive_mean).masked_fill(~usable, 0.0)
    return per_anchor.sum() / usable.sum().clamp_min(1), usable.sum()


class CosineHead(nn.Module):
    def __init__(self, width, classes):
        super().__init__()
        self.weight = nn.Parameter(torch.empty(classes, width))
        nn.init.normal_(self.weight, std=0.02)

    def forward(self, features):
        return LOGIT_SCALE * F.linear(F.normalize(features, dim=1), F.normalize(self.weight, dim=1))


class ConceptHeads(nn.Module):
    def __init__(self, width, prefix_tokens):
        super().__init__()
        self.prefix_tokens = prefix_tokens
        self.attention = nn.Sequential(nn.Linear(width, 128), nn.Tanh(), nn.Linear(128, 1, bias=False))
        nn.init.zeros_(self.attention[-1].weight)
        self.disease_projection = nn.Sequential(nn.LayerNorm(width), nn.Linear(width, 256))
        self.crop_head = CosineHead(width, NUM_CROPS)
        self.disease_head = CosineHead(256, NUM_DISEASES)

    def forward(self, tokens, return_patches=False):
        crop_feature = tokens[:, 0]
        patches = tokens[:, self.prefix_tokens:]
        attention = self.attention(patches).softmax(dim=1)
        disease_feature = self.disease_projection((patches * attention).sum(dim=1))
        outputs = (self.crop_head(crop_feature), self.disease_head(disease_feature), disease_feature)
        return (*outputs, patches) if return_patches else outputs


class PatchPairNet(nn.Module):
    def __init__(self, pretrained):
        super().__init__()
        self.backbone = timm.create_model(BACKBONE, pretrained=pretrained, num_classes=0,
                                          img_size=team.IMAGE_SIZE)
        self.backbone.requires_grad_(False)
        self.heads = ConceptHeads(self.backbone.num_features, self.backbone.num_prefix_tokens)

    def forward(self, images):
        return self.heads(self.backbone.forward_features(images))


class FineTuneTail(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.block = copy.deepcopy(model.backbone.blocks[-1]).requires_grad_(True)
        self.norm = copy.deepcopy(model.backbone.norm).requires_grad_(True)
        self.heads = copy.deepcopy(model.heads)

    def forward(self, tokens, return_patches=False):
        return self.heads(self.norm(self.block(tokens)), return_patches=return_patches)


@torch.no_grad()
def verify_prefix(model, tail, images):
    for split, regular in zip(tail(frozen_prefix(model.backbone, images)), model(images)):
        torch.testing.assert_close(split, regular, rtol=1e-5, atol=1e-6)


def concept_pair_scores(crop_logits, disease_logits, pairs):
    return crop_logits[:, pairs[:, 0]] + disease_logits[:, pairs[:, 1]]


class PairRankingObjective(nn.Module):
    def __init__(self, fit_pair_rows, pairs):
        super().__init__()
        count = torch.bincount(fit_pair_rows, minlength=len(pairs))
        active = count.gt(0)
        if active.sum().item() < 2:
            raise ValueError("Pair ranking needs at least two fit pairs.")
        self.register_buffer("active", active)
        self.register_buffer("pairs", pairs)
        self.register_buffer("weights", count.clamp_min(1).float().pow(-0.5))
        self.hard_count = min(HARD_NEGATIVES, int(active.sum()) - 1)

    def margins(self, crop, disease, pair_rows):
        scores = concept_pair_scores(crop, disease, self.pairs)
        target = scores.gather(1, pair_rows[:, None])
        negatives = scores.masked_fill(~self.active[None, :], -torch.inf)
        negatives = negatives.scatter(1, pair_rows[:, None], -torch.inf)
        hardest = negatives.topk(self.hard_count, dim=1).values
        return F.softplus(PAIR_MARGIN + hardest - target).mean(dim=1)

    def forward(self, crop, disease, crop_labels, disease_labels, pair_rows):
        weights = self.weights[pair_rows]
        weights = weights / weights.sum()
        crop_ce = F.cross_entropy(crop, crop_labels, label_smoothing=SMOOTHING)
        disease_ce = F.cross_entropy(disease, disease_labels, reduction="none", label_smoothing=SMOOTHING)
        real_rank = torch.dot(weights, self.margins(crop, disease, pair_rows))

        # Attach disease evidence from same-disease images of other crops to the anchor crop.
        # The supervised target remains the anchor's real fit pair.
        donors = disease_labels[:, None].eq(disease_labels) & crop_labels[:, None].ne(crop_labels)
        donor_counts = donors.sum(dim=1)
        recombined = (donors.to(disease.dtype) @ disease) / donor_counts.clamp_min(1)[:, None]
        available = donor_counts > 0
        transfer_weights = self.weights[pair_rows] * available
        transfer_weights = transfer_weights / transfer_weights.sum().clamp_min(1e-8)
        transfer_rank = torch.dot(transfer_weights, self.margins(crop, recombined, pair_rows))

        total = crop_ce + torch.dot(weights, disease_ce) + RANK_WEIGHT * real_rank + TRANSFER_WEIGHT * transfer_rank
        return total, real_rank, transfer_rank, available.sum()

def paired_patch_loss(heads, patches, student, crops, diseases, grid_size):
    eligible = diseases[:, None].eq(diseases[None, :]) & crops[:, None].ne(crops[None, :])
    anchors, donors = eligible.nonzero(as_tuple=True)
    if not len(anchors):
        return student.new_zeros(()), 0
    # Average donor evidence per anchor, so crops with many eligible donors do not gain extra weight.
    width = patches.shape[2]
    spatial = patches.transpose(1, 2).reshape(len(patches), width, *grid_size)
    spatial = F.adaptive_avg_pool2d(spatial, tuple(min(PATCH_GRID, size) for size in grid_size))
    spatial = spatial.flatten(2).transpose(1, 2)
    normalized = F.normalize(spatial.detach(), dim=-1)
    similarities = normalized[anchors] @ normalized[donors].transpose(1, 2)
    matches = (similarities.amax(dim=2) / DONOR_TEMPERATURE).softmax(dim=1)
    pooled = (matches.unsqueeze(-1) * spatial[anchors]).sum(dim=1)
    shared = heads.disease_projection(pooled)
    ce = F.cross_entropy(heads.disease_head(shared), diseases[anchors], reduction="none",
                         label_smoothing=SMOOTHING)
    alignment = 1 - F.cosine_similarity(student[anchors], shared.detach(), dim=1)
    counts = eligible.sum(dim=1)
    weights = counts[anchors].float().reciprocal()
    total = (weights * (PAIRED_WEIGHT * ce + ALIGNMENT_WEIGHT * alignment)).sum() / weights.sum()
    return total, int(counts.count_nonzero())

## 4. Training

Train both arms from identical weights with one frozen-prefix forward pass per batch. Choose the recipe and epoch by raw tune harmonic mean, then proxy Top-1, seen Top-1 and earlier epoch. Select gamma separately using the tune split. The check split measures performance after that selection. `raw_check_deltas.csv` shows paired A5 minus A5 control at the selected winner's epoch. Track raw proxy Top-1, harmonic mean and disease discrimination with crop supplied for diagnosis.

In [ ]:
def selection_key(row):
    return (row["harmonic_top1"], row["proxy_top1"], row["seen_top1"], -row["epoch"])


def snapshot_tail(tail):
    return {name: value.detach().cpu().clone() for name, value in tail.state_dict().items()}


def make_optimizer(tail):
    return torch.optim.AdamW([
        {"params": tail.heads.parameters(), "lr": HEAD_LR},
        {"params": [*tail.block.parameters(), *tail.norm.parameters()], "lr": BACKBONE_LR},
    ], weight_decay=WEIGHT_DECAY)


def train_tail(recipe, tail, optimizer, objective, tokens, crop, disease, pair_rows, grid_size):
    extra = RECIPES[recipe]
    outputs = tail(tokens, return_patches=extra)
    crop_logits, disease_logits, disease_features = outputs[:3]
    pair_loss, ranking, transfer, transfers = objective(crop_logits, disease_logits, crop, disease, pair_rows)
    decor = decorrelation(center_logits(crop_logits), center_logits(disease_logits))
    supcon, anchors = cross_crop_supcon(disease_features, crop, disease)
    loss = pair_loss + DECOR_WEIGHT * decor + SUPCON_WEIGHT * supcon
    aux, donor_count = loss.new_zeros(()), 0
    if extra:
        aux, donor_count = paired_patch_loss(tail.heads, outputs[3], disease_features, crop, disease, grid_size)
        loss = loss + aux
    if not torch.isfinite(loss):
        raise FloatingPointError(f"Non-finite training loss in {recipe}.")
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    return (loss.item(), ranking.item(), transfer.item(), int(anchors), int(transfers), aux.item(), donor_count)


@torch.no_grad()
def cached_group(backbone, frame):
    rng = torch.Generator().manual_seed(team.SEED)
    loader = DataLoader(PlantDataset(frame, image_transform, "imagenet"), batch_size=BATCH_SIZE,
                        shuffle=False, num_workers=0, generator=rng)
    return [(frozen_prefix(backbone, images.to(DEVICE)).cpu(), target.cpu())
            for images, _, _, target in loader]


@torch.no_grad()
def score_group(tail, batches):
    scores, labels = [], []
    for tokens, targets in batches:
        crop, disease, _ = tail(tokens.to(DEVICE))
        scores.append(concept_pair_scores(crop, disease, train_pairs_device).cpu())
        labels.append(targets)
    return torch.cat(scores), torch.cat(labels)


def metrics(seen, proxy, fit_mask, gamma=0.0):
    seen_s, seen_y = seen
    proxy_s, proxy_y = proxy
    seen_s, proxy_s = (value - gamma * fit_mask for value in (seen_s, proxy_s))
    s, u = top_k_accuracy(seen_s, seen_y, 1), top_k_accuracy(proxy_s, proxy_y, 1)
    winner = proxy_s.argmax(1)
    held_winner = proxy_s.masked_fill(fit_mask[None, :], -torch.inf).argmax(1)
    blocked = held_winner.eq(proxy_y) & fit_mask[winner]
    eligible = train_pairs[proxy_y, 0, None].eq(train_pairs[None, :, 0])
    same_crop_top1 = top_k_accuracy(proxy_s.masked_fill(~eligible, -torch.inf), proxy_y, 1)
    return {"seen_top1": s, "proxy_top1": u, "harmonic_top1": harmonic_mean(s, u),
            "seen_top5": top_k_accuracy(seen_s, seen_y, 5),
            "proxy_top5": top_k_accuracy(proxy_s, proxy_y, 5),
            "proxy_blocked_by_fit_pairs_pct": round(100 * float(blocked.float().mean()), 2),
            "proxy_true_crop_top1": same_crop_top1}


def read_final_lock():
    if not LOCK.is_file() or not CHECKPOINT.is_file():
        raise FileNotFoundError("Complete A5 training first.")
    saved = json.loads(LOCK.read_text(encoding="utf-8"))
    if saved["settings"] != settings or saved["checkpoint_sha256"] != sha256_file(CHECKPOINT):
        raise RuntimeError("A5 checkpoint or settings changed after selection.")
    if saved["seen_pairs"] != train_pairs.tolist():
        raise RuntimeError("Training candidate identities changed after selection.")
    return saved


def objective_for(frame):
    targets = torch.as_tensor(frame.pair_row.to_numpy(copy=True), dtype=torch.long)
    return PairRankingObjective(targets, train_pairs).to(DEVICE)


def make_training_model():
    model = PatchPairNet(pretrained=True).to(DEVICE).eval()
    if len(model.backbone.blocks) != 12:
        raise ValueError("Expected the 12-block DINOv2 ViT-B/14.")
    tail = FineTuneTail(model).to(DEVICE).eval()
    return model, tail


def train_epoch(model, trials, objective, loader, blur_rng, stage, epoch, epochs, fit_mask=None):
    sums = {recipe: [0.0] * 7 for recipe in trials}
    examples = 0
    for step, (images, crops, diseases, pair_rows) in enumerate(loader, 1):
        images = augment_blur(images.to(DEVICE), blur_rng)
        crops, diseases, pair_rows = (tensor.to(DEVICE) for tensor in (crops, diseases, pair_rows))
        if fit_mask is not None:
            keep = fit_mask[pair_rows]
            if not keep.any():
                continue
            images, crops, diseases, pair_rows = images[keep], crops[keep], diseases[keep], pair_rows[keep]
        if epoch == 1 and examples == 0:
            verify_prefix(model, next(iter(trials.values()))[0], images[:1])
        tokens = frozen_prefix(model.backbone, images)
        for recipe, (tail, optimizer) in trials.items():
            values = train_tail(recipe, tail, optimizer, objective, tokens, crops, diseases, pair_rows,
                                model.backbone.patch_embed.grid_size)
            loss, rank, transfer, anchors, transfers, aux, donors = values
            add = (loss * len(images), rank * len(images), transfer * transfers, anchors,
                   transfers, aux * donors, donors)
            for i, value in enumerate(add):
                sums[recipe][i] += value
        examples += len(images)
        if step == 1 or step % 50 == 0 or step == len(loader):
            print(f"A5 {stage} epoch {epoch}/{epochs} · batch {step}/{len(loader)}", flush=True)
    if examples == 0:
        raise ValueError("No fit images were processed.")
    return {recipe: {"recipe": recipe, "epoch": epoch, "images_trained": examples,
                     "train_loss": loss / examples, "train_ranking_loss": rank / examples,
                     "train_transfer_ranking_loss": transfer / max(transfers, 1),
                     "transfer_anchor_fraction": transfers / examples, "supcon_anchor_fraction": anchors / examples,
                     "paired_loss": aux / max(donors, 1), "paired_anchor_fraction": donors / examples}
            for recipe, (loss, rank, transfer, anchors, transfers, aux, donors) in sums.items()}


def run_development():
    DEVELOPMENT_DIR.mkdir(parents=True, exist_ok=True)
    atomic_json(DEVELOPMENT_DIR / "environment.json", environment)
    (DEVELOPMENT_DIR / "requirements.txt").write_bytes(requirements_bytes)
    team.set_seed()
    train, val, test_seen, test_unseen = get_loaders(BATCH_SIZE, norm="imagenet", debug=DEBUG)
    if test_seen is not None or test_unseen is not None:
        raise RuntimeError("Development must use only the training CSV.")
    train.pin_memory = False
    model, control = make_training_model()
    tails = {CONTROL: control, PAIRED: copy.deepcopy(control)}
    trials = {recipe: (tail, make_optimizer(tail)) for recipe, tail in tails.items()}
    schedules = [torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
                 for _, optimizer in trials.values()]
    fit_frame, fit_mask, fit_device = train.dataset.frame, None, None
    if not DEBUG:
        fit_mask, frames = proxy_split(fit_frame, val.dataset.frame)
        fit_device = fit_mask.to(DEVICE)
        fit_frame = fit_frame.loc[fit_frame.pair_row.isin(torch.where(fit_mask)[0].tolist())]
        tune_cache = [cached_group(model.backbone, frames[key]) for key in ("tune_seen", "tune_proxy")]
    objective = objective_for(fit_frame)
    if fit_mask is not None and not torch.equal(objective.active, fit_device):
        raise RuntimeError("Proxy holdout and fit pairs do not match.")

    blur_rng = torch.Generator().manual_seed(team.SEED + 172)
    best, history = {}, []
    for epoch in range(1, EPOCHS + 1):
        rows = train_epoch(model, trials, objective, train, blur_rng, "development", epoch, EPOCHS, fit_device)
        for schedule in schedules:
            schedule.step()
        if not DEBUG:
            tune = {recipe: [score_group(tail, cache) for cache in tune_cache] for recipe, tail in tails.items()}
            improving = []
            for recipe, row in rows.items():
                row.update(metrics(*tune[recipe], fit_mask))
                if recipe not in best or selection_key(row) > selection_key(best[recipe]["selection"]):
                    improving.append(recipe)
            if improving:
                states = {recipe: snapshot_tail(tail) for recipe, tail in tails.items()}
                for recipe in improving:
                    best[recipe] = {"selection": rows[recipe].copy(), "states": states, "scores": tune[recipe]}
        history.extend(rows.values())
        for row in rows.values():
            print(row, flush=True)

    pd.DataFrame(history).to_csv(DEVELOPMENT_DIR / "training_history.csv", index=False)
    if DEBUG:
        return None
    if not any(row["paired_anchor_fraction"] > 0 for row in history):
        raise RuntimeError("Paired supervision did not see an eligible training pair.")
    winner = max(best, key=lambda recipe: selection_key(best[recipe]["selection"]))
    del tune_cache
    check_cache = [cached_group(model.backbone, frames[key]) for key in ("check_seen", "check_proxy")]
    checks, grid, selections = [], [], {}
    for recipe, record in best.items():
        selection = record["selection"]
        candidates = [{"recipe": recipe, "gamma": gamma, **metrics(*record["scores"], fit_mask, gamma)} for gamma in GAMMAS]
        gamma = max(candidates, key=lambda row: (row["harmonic_top1"], row["proxy_top1"],
                                                 row["seen_top1"], -row["gamma"]))["gamma"]
        grid.extend(candidates)
        tail = tails[recipe]
        tail.load_state_dict(record["states"][recipe])
        scores = [score_group(tail, cache) for cache in check_cache]
        checks.extend({"recipe": recipe, "epoch": selection["epoch"], "scoring": name, "gamma": g,
                       **metrics(*scores, fit_mask, g)}
                      for name, g in (("raw", 0.0), ("calibrated", gamma)))
        selections[recipe] = {"selection": selection, "gamma": gamma}
    matched = {}
    for recipe, tail in tails.items():
        tail.load_state_dict(best[winner]["states"][recipe])
        matched[recipe] = metrics(*(score_group(tail, cache) for cache in check_cache), fit_mask)
    delta = {f"{key}_delta_pp": round(matched[PAIRED][key] - matched[CONTROL][key], 2) for key in matched[CONTROL]}
    pd.DataFrame([{"epoch": best[winner]["selection"]["epoch"], "recipe": PAIRED, "reference": CONTROL,
                   **delta}]).to_csv(DEVELOPMENT_DIR / "raw_check_deltas.csv", index=False)
    pd.DataFrame([{"recipe": recipe, **values} for recipe, values in matched.items()]).to_csv(
        DEVELOPMENT_DIR / "raw_matched_check.csv", index=False)
    pd.DataFrame(checks).to_csv(DEVELOPMENT_DIR / "independent_proxy_check.csv", index=False)
    pd.DataFrame(grid).to_csv(DEVELOPMENT_DIR / "calibration_tune_grid.csv", index=False)
    table = pd.DataFrame([{**value["selection"], "locked_gamma": value["gamma"], "selected": recipe == winner}
                          for recipe, value in selections.items()])
    table.to_csv(DEVELOPMENT_DIR / "selected_models.csv", index=False)
    plan = {"recipe": winner, **selections[winner], "recipes": selections,
            "tune_proxy_pairs": sorted(frames["tune_proxy"].pair_row.unique().tolist()),
            "check_proxy_pairs": sorted(frames["check_proxy"].pair_row.unique().tolist())}
    atomic_json(DEVELOPMENT_DIR / "epoch_selection.json", plan)
    display(table)
    return plan


development = run_development()

## 5. Final evaluation preparation

Refit only the winner on every training row from the pretrained backbone and fresh heads. Record the refit history and lock the selected training plan, gamma and checkpoint hash before loading official metadata.

In [ ]:
def refit_full_train(development):
    if LOCK.is_file():
        locked = read_final_lock()
        if locked["development"] != development:
            raise RuntimeError("Existing checkpoint belongs to a different selection.")
        return locked
    if json.loads((DEVELOPMENT_DIR / "epoch_selection.json").read_text(encoding="utf-8")) != development:
        raise RuntimeError("The development plan changed before refit.")
    if sha256_file(team.TRAIN_CSV) != settings["train_csv_sha256"]:
        raise RuntimeError("Training CSV changed after development. Start a fresh run.")

    selected_epochs = int(development["selection"]["epoch"])
    if not 1 <= selected_epochs <= EPOCHS:
        raise ValueError("Selected refit length is outside the development epoch budget.")
    team.set_seed()
    model, tail = make_training_model()
    recipe = development["recipe"]
    optimizer = make_optimizer(tail)
    schedule = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
    train_transform, _ = get_transforms("imagenet")
    loader = DataLoader(PlantDataset(train_full, train_transform, "imagenet"), batch_size=BATCH_SIZE,
                        shuffle=True, num_workers=team.NUM_WORKERS, drop_last=False,
                        generator=torch.Generator().manual_seed(team.SEED))
    objective = objective_for(train_full)
    if not objective.active.all():
        raise RuntimeError("Full training data must cover all training candidate pairs.")

    blur_rng = torch.Generator().manual_seed(team.SEED + 172)
    history = []
    for epoch in range(1, selected_epochs + 1):
        row = train_epoch(model, {recipe: (tail, optimizer)}, objective, loader, blur_rng,
                          "full refit", epoch, selected_epochs)[recipe]
        if row["images_trained"] != len(train_full):
            raise RuntimeError("A refit epoch did not process every training image.")
        schedule.step()
        history.append(row)
        print(row, flush=True)

    model.backbone.blocks[-1].load_state_dict(tail.block.state_dict())
    model.backbone.norm.load_state_dict(tail.norm.state_dict())
    model.heads.load_state_dict(tail.heads.state_dict())
    model.cpu()
    refit = {"initialization": "original_pretrained_dinov2_fresh_heads",
             "data": "all_training_csv_rows", "images_per_epoch": len(train_full),
             "training_pairs": len(train_pairs), "epochs": selected_epochs, "cosine_t_max": EPOCHS}
    checkpoint_metadata = {"settings": settings, "recipe": recipe, "selection": development["selection"],
                           "gamma": development["gamma"], "paired_supervision": RECIPES[recipe],
                           "seen_pairs": train_pairs.tolist(), "refit": refit}
    CHECKPOINT.parent.mkdir(parents=True, exist_ok=True)
    FINAL_DIR.mkdir(parents=True, exist_ok=True)
    pd.DataFrame(history).to_csv(FINAL_DIR / "refit_history.csv", index=False)
    atomic_save(CHECKPOINT, {"state_dict": model.state_dict(), **checkpoint_metadata})
    locked = {"development": development, **checkpoint_metadata, "checkpoint_sha256": sha256_file(CHECKPOINT),
              "checkpoint_bytes": CHECKPOINT.stat().st_size}
    atomic_json(LOCK, locked)
    return locked


run = None if DEBUG else refit_full_train(development)
if run is not None:
    print(f"Full refit locked: {run['refit']['images_per_epoch']} images per epoch, "
          f"{run['refit']['epochs']} epochs.")

## 6. Official test

Generate 333-way raw and calibrated predictions before joining per-image ground truth. Raw uses `gamma=0` and is the primary result. The supplementary row applies the train-only locked seen penalty. Report both, along with robustness and per-pair ranking diagnostics.

In [ ]:
def official_vocabulary():
    # The final shortlist uses only concept identities and discards per-image row order.
    vocabulary = pd.read_csv(team.TEST_UNSEEN_CSV, header=None, usecols=[1, 2],
                             dtype=str, keep_default_na=False)
    vocabulary.columns = ["crop", "disease"]
    if not vocabulary.empty and not vocabulary.iloc[0]["crop"].strip().isdigit():
        vocabulary = vocabulary.iloc[1:]
    vocabulary = (vocabulary.astype("int64").drop_duplicates()
                  .sort_values(["crop", "disease"]))
    extra_pairs = torch.as_tensor(vocabulary.to_numpy(copy=True), dtype=torch.long)
    pairs = torch.cat((train_pairs, extra_pairs))
    if (len(extra_pairs) != team.NUM_UNSEEN_PAIRS or len(pairs.unique(dim=0)) != team.NUM_PAIRS
            or not set(extra_pairs[:, 0].tolist()) <= set(train_pairs[:, 0].tolist())
            or not set(extra_pairs[:, 1].tolist()) <= set(train_pairs[:, 1].tolist())):
        raise ValueError("Expected distinct official unseen pairs composed of training-derived concepts.")
    return pairs


def official_filenames(path):
    values = pd.read_csv(path, header=None, usecols=[0], dtype=str, keep_default_na=False).iloc[:, 0].tolist()
    header = {"image", "image_path", "image_name", "filename", "file_name", "file_path", "path"}
    if values and (values[0].strip().lower().replace(" ", "_") in header or
                   (len(values) > 1 and not Path(values[0]).suffix and Path(values[1]).suffix)):
        values = values[1:]
    names = [image_key(name) for name in values]
    if not names or len(set(names)) != len(names):
        raise ValueError(f"Expected unique image names in {path.name}.")
    return names


class OfficialImages(Dataset):
    def __init__(self, names):
        self.names = names

    def __len__(self):
        return len(self.names)

    def __getitem__(self, index):
        with Image.open(team.IMAGE_DIR / self.names[index]) as image:
            return image_transform(image.convert("RGB"))


def official_targets(csv_path, filenames, row_by_pair, *, seen):
    # This per-image projection is used after predictions. Ignore the official pair_label column.
    frame = pd.read_csv(csv_path, header=None, usecols=[0, 1, 2], dtype=str, keep_default_na=False)
    frame.columns = ["image", "crop", "disease"]
    if not frame.empty and not frame.iloc[0]["crop"].strip().isdigit():
        frame = frame.iloc[1:]
    if frame.image.map(image_key).tolist() != filenames:
        raise ValueError("Official targets and predictions have different image orders.")
    keys = frame[["crop", "disease"]].astype("int64").itertuples(index=False, name=None)
    try:
        labels = torch.tensor([row_by_pair[key] for key in keys], dtype=torch.long)
    except KeyError as error:
        raise ValueError(f"Official ground-truth pair absent from the frozen candidate table: {error}") from error
    if not ((labels < team.NUM_SEEN_PAIRS) == seen).all():
        raise ValueError("Official test split does not match seen membership.")
    return labels


def top_predictions(scores, offset=0):
    return {k: scores.topk(k, dim=1).indices + offset for k in (1, 5)}


def prediction_accuracy(predictions, labels):
    return round(100 * predictions.eq(labels[:, None]).any(1).float().mean().item(), 2)


@torch.inference_mode()
def predict_official(model, pairs, filenames, split, perturb=None):
    generator = torch.Generator().manual_seed(team.SEED)
    loader = DataLoader(OfficialImages(filenames), batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=0, generator=generator)
    mean, std = (torch.tensor(values).view(1, 3, 1, 1) for values in team.NORM_STATS["imagenet"])
    pair_indices = pairs.to(DEVICE)
    scores, concepts = [], []
    for step, images in enumerate(loader, 1):
        if perturb is not None:
            images = (perturb(images * std + mean) - mean) / std
        crop, disease, _ = model(images.to(DEVICE))
        batch = concept_pair_scores(crop, disease, pair_indices).cpu()
        if batch.shape != (len(images), len(pairs)) or not torch.isfinite(batch).all():
            raise ValueError("Invalid raw pair scores.")
        scores.append(batch)
        concepts.append(torch.stack((crop.argmax(dim=1), disease.argmax(dim=1)), dim=1).cpu())
        if step == 1 or step % 50 == 0 or step == len(loader):
            print(f"{split}: batch {step}/{len(loader)}", flush=True)
    return torch.cat(scores), torch.cat(concepts)


def predict_recipe(recipe, run, pairs, filenames):
    saved = torch.load(CHECKPOINT, map_location="cpu", weights_only=True)
    for key in ("settings", "recipe", "paired_supervision", "selection", "gamma", "seen_pairs", "refit"):
        if saved[key] != run[key]:
            raise RuntimeError(f"Checkpoint and selection lock differ for {recipe}: {key}.")
    model = PatchPairNet(pretrained=False)
    model.load_state_dict(saved["state_dict"], strict=True)
    model = model.to(DEVICE).eval()
    boundary = len(train_pairs)
    membership = torch.arange(len(pairs)) < boundary
    clean, heads = zip(*(predict_official(model, pairs, names, split)
                         for names, split in zip(filenames, ("seen", "unseen"))))
    perturbed = {}
    with torch.random.fork_rng(devices=[]):
        torch.random.default_generator.manual_seed(team.SEED)
        for condition, transform in PERTURBATIONS.items():
            perturbed[condition], _ = predict_official(model, pairs, filenames[0], condition, transform)

    predictions = {}
    for rule, gamma in (("raw", 0.0), ("calibrated", run["gamma"])):
        seen_scores, unseen_scores = (scores - gamma * membership for scores in clean)
        predictions[rule] = {
            "gamma": gamma,
            "seen": top_predictions(seen_scores),
            "unseen": top_predictions(unseen_scores),
            "seen_only": top_predictions(seen_scores[:, :boundary]),
            "unseen_only": top_predictions(unseen_scores[:, boundary:], offset=boundary),
            "robust": {name: (scores - gamma * membership).topk(1, dim=1).indices
                       for name, scores in perturbed.items()},
        }

    return predictions, clean, heads


def score_trial(run, pairs, forward_pass, y_seen, y_unseen):
    predictions, clean, (seen_heads, unseen_heads) = forward_pass
    boundary = len(train_pairs)
    y_all = torch.cat((y_seen, y_unseen))

    reports, ranking, per_pair, robustness, restricted = [], [], [], [], []
    # Rows are unseen images and columns are candidate pairs.
    true_crop = pairs[y_unseen, 0]
    same_crop_candidates = true_crop[:, None].eq(pairs[None, :, 0])
    seen_head_hits = seen_heads.eq(pairs[y_seen])
    unseen_head_hits = unseen_heads.eq(pairs[y_unseen])
    head_top1 = {
        f"{split}_{head}_evidence_top1": round(100 * matches[:, col].float().mean().item(), 2)
        for split, matches in (("seen", seen_head_hits), ("unseen", unseen_head_hits))
        for col, head in enumerate(("crop", "disease"))
    }
    for name, prediction in predictions.items():
        gamma = prediction["gamma"]
        raw_scores = clean[1]
        crop_oracle = raw_scores.masked_fill(~same_crop_candidates, -torch.inf).topk(1, dim=1).indices
        raw_margins = raw_scores.gather(1, y_unseen[:, None]).squeeze(1) - raw_scores[:, :boundary].amax(dim=1)
        final_margins = raw_margins + gamma
        only_correct = prediction["unseen_only"][1].squeeze(1).eq(y_unseen)
        metrics = {"scoring": name, "gamma": gamma, "paired_supervision": run["paired_supervision"],
                   "selected_on_tune": run["recipe"] == development["recipe"],
                   "epoch": run["selection"]["epoch"], "candidates": len(pairs), **head_top1,
                   "raw_true_crop_candidate_top1": prediction_accuracy(crop_oracle, y_unseen)}
        for k in (1, 5):
            metrics[f"seen_top{k}"] = prediction_accuracy(prediction["seen"][k], y_seen)
            metrics[f"unseen_top{k}"] = prediction_accuracy(prediction["unseen"][k], y_unseen)
            joined = torch.cat((prediction["seen"][k], prediction["unseen"][k]))
            metrics[f"overall_top{k}"] = prediction_accuracy(joined, y_all)
        metrics["harmonic_mean"] = harmonic_mean(metrics["seen_top1"], metrics["unseen_top1"])
        metrics["abs_seen_unseen_gap_pp"] = round(abs(metrics["seen_top1"] - metrics["unseen_top1"]), 2)
        seen_winner, unseen_winner = (prediction[split][1].squeeze(1) for split in ("seen", "unseen"))
        metrics["seen_predicted_unseen"] = int((seen_winner >= boundary).sum())
        for split, winner, target in (("seen", seen_winner, y_seen), ("unseen", unseen_winner, y_unseen)):
            concept_hits = pairs[winner].eq(pairs[target]).float().mean(dim=0) * 100
            metrics[f"{split}_test_crop_acc"], metrics[f"{split}_test_disease_acc"] = [
                round(float(value), 2) for value in concept_hits]
        reports.append(metrics)

        correct = unseen_winner.eq(y_unseen)
        predicts_seen = unseen_winner < boundary
        blocked = only_correct & predicts_seen
        same_crop = pairs[unseen_winner, 0].eq(pairs[y_unseen, 0])
        ranking.append({"scoring": name, "unseen_images": len(y_unseen), "joint_correct": int(correct.sum()),
                        "unseen_only_correct": int(only_correct.sum()), "unseen_predicted_seen": int(predicts_seen.sum()),
                        "blocked_by_seen": int(blocked.sum()), "blocked_same_crop": int((blocked & same_crop).sum()),
                        "blocked_other_crop": int((blocked & ~same_crop).sum()),
                        "median_true_pair_minus_top_seen_score": float(final_margins.median())})
        for index in range(boundary, len(pairs)):
            mask = y_unseen.eq(index)
            crop, disease = pairs[index].tolist()
            per_pair.append({"scoring": name, "crop": crop, "disease": disease, "support": int(mask.sum()),
                             "joint_correct": int((mask & correct).sum()),
                             "unseen_only_correct": int((mask & only_correct).sum()),
                             "blocked_by_seen": int((mask & blocked).sum())})
        for key, predictions_at_one in prediction["robust"].items():
            accuracy = prediction_accuracy(predictions_at_one, y_seen)
            robustness.append({"scoring": name, "condition": key, "seen_top1": accuracy,
                               "drop_pp": round(metrics["seen_top1"] - accuracy, 2)})
        for split, targets, count in (("seen_only", y_seen, boundary),
                ("unseen_only", y_unseen, len(pairs) - boundary)):
            restricted.append({"scoring": name, "split": split, "candidates": count,
                               **{f"top{k}": prediction_accuracy(prediction[split][k], targets) for k in (1, 5)}})
    return ({"results": reports, "restricted_candidates": restricted, "robustness": robustness,
             "unseen_ranking_diagnostics": ranking, "unseen_per_pair": per_pair},
            {name: pred["seen"][1].squeeze(1) for name, pred in predictions.items()})


def official_evaluation():
    if DEBUG or run is None:
        raise RuntimeError("Complete the selected model's full refit before official evaluation.")
    recipe = development["recipe"]
    if read_final_lock() != run:
        raise RuntimeError("The refitted checkpoint differs from the selected lock.")
    pairs = official_vocabulary()
    paths = (team.TEST_SEEN_CSV, team.TEST_UNSEEN_CSV)
    filenames = tuple(official_filenames(path) for path in paths)
    seen_names, unseen_names = map(set, filenames)
    if seen_names & unseen_names or set(train_full.image) & (seen_names | unseen_names):
        raise ValueError("Training and official split filenames overlap.")
    predictions = predict_recipe(recipe, run, pairs, filenames)
    rows = {tuple(pair): i for i, pair in enumerate(pairs.tolist())}
    y_seen = official_targets(paths[0], filenames[0], rows, seen=True)
    y_unseen = official_targets(paths[1], filenames[1], rows, seen=False)
    report, predicted_seen = score_trial(run, pairs, predictions, y_seen, y_unseen)
    summary = {"selection": run, "recipe_selected_on_tune": recipe,
               "samples": {"seen": len(y_seen), "unseen": len(y_unseen)},
               "pair_order": "train_then_concept_ids", "candidate_pairs": pairs.tolist()}
    for key, values in report.items():
        summary[key] = [{"recipe": recipe, **row} for row in values]
    winners = {f"{recipe}_{rule}": values for rule, values in predicted_seen.items()}
    return summary, winners, y_seen


evaluation = None if DEBUG else official_evaluation()

## 7. Save

- `results/approach2_a5_paired_attention/<run_id>/development/` saves the controlled comparisons and environment.
- `final/` saves the refit log and selection lock.
- `official/` saves raw and calibrated metrics, robustness, confusion matrices and ranking diagnostics.
- `checkpoints/approach2_a5_paired_attention/<run_id>/model.pt` saves the complete winning model.

Only the selected arm is tested officially in this run. Label earlier official results as historical references.

In [ ]:
def save_results(summary, seen_winners, seen_targets):
    OFFICIAL_DIR.mkdir(parents=True, exist_ok=True)
    # Align the team confusion plot order only after predictions and metrics are fixed.
    pairs = [tuple(pair) for pair in summary["candidate_pairs"]]
    lookup = {tuple(pair): row for row, pair in enumerate(get_pairs(final_evaluation=True).tolist())}
    if set(pairs) != lookup.keys():
        raise ValueError("Candidate identities differ from the common evaluator.")
    to_common = torch.tensor([lookup[pair] for pair in pairs])
    canonical_targets = to_common[seen_targets]
    if (canonical_targets >= len(train_pairs)).any():
        raise ValueError("The common confusion plot expects seen target rows.")
    for name, winners in seen_winners.items():
        votes = F.one_hot(to_common[winners], num_classes=len(pairs)).float()
        stem = (OFFICIAL_DIR / name).relative_to(team.RESULTS_DIR).as_posix()
        save_confusion(votes, canonical_targets, stem, final_evaluation=True)

    table = pd.DataFrame(summary["results"])
    table.to_csv(OFFICIAL_DIR / "official_comparison.csv", index=False)
    for mode, part in table.groupby("scoring", sort=False):
        part.to_csv(OFFICIAL_DIR / f"official_{mode}_results.csv", index=False)
    for name in ("restricted_candidates", "robustness", "unseen_ranking_diagnostics", "unseen_per_pair"):
        pd.DataFrame(summary[name]).to_csv(OFFICIAL_DIR / f"{name}.csv", index=False)
    for row in summary["results"]:
        atomic_json(OFFICIAL_DIR / f"{row['recipe']}_{row['scoring']}.json", row)
    atomic_json(OFFICIAL_DIR / "official_report.json", summary)
    return table


if evaluation is not None:
    table = save_results(*evaluation)
    columns = ["recipe", "paired_supervision", "gamma", "epoch", "selected_on_tune",
               "seen_top1", "unseen_top1", "overall_top1", "seen_top5", "unseen_top5", "overall_top5", "harmonic_mean"]
    for mode, part in table.groupby("scoring", sort=False):
        print(f"{mode.capitalize()} results")
        display(part[columns])
    recipe = development["recipe"]
    print(recipe, "checkpoint:", CHECKPOINT.resolve())
    print("Development results:", DEVELOPMENT_DIR.resolve())
    print("Final selection and refit:", FINAL_DIR.resolve())
    print("Official results:", OFFICIAL_DIR.resolve())
else:
    print("Debug logs:", DEVELOPMENT_DIR.resolve())